In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Colab용 환경설정
# 필요한 경우 opencv 설치 (Colab에는 기본적으로 설치되어 있으나 확인용)
!pip install opencv-python

In [ ]:
# Colab용 환경설정
# 터미널 명령어로 원하는 경로(-p)에 kaggle 데이터셋 다운로드 및 압축 해제(--unzip)
# 데이터셋이 없는 경우 다운로드 용도!
!kaggle datasets download -d inwoohwang2/parking-vehicle-hit-an-run-dataset -p /content/drive/MyDrive/capstone-26 --unzip

In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms

# ==========================================
# [1단계] 데이터 전처리 및 Custom Dataset
# ==========================================
class HitAndRunDataset(Dataset):
    def __init__(self, data_dir, clip_length=30, r_value=1.0, resize=(224, 224)):
        self.data_dir = data_dir
        self.clip_length = clip_length
        self.r_value = r_value
        self.resize = resize

        self.file_names = [f.split('.')[0] for f in os.listdir(data_dir) if f.endswith('.mp4')]

        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.file_names)

    def _parse_annotation(self, txt_path):
        bboxes = {}
        action = None
        if os.path.exists(txt_path):
            with open(txt_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split(',')
                    if not parts or len(parts) < 2: continue
                    if parts[0] == 'car':
                        bboxes[int(parts[1])] = [int(parts[2]), int(parts[3]), int(parts[4]), int(parts[5])]
                    elif parts[0] in ['A', 'S']:
                        action = parts
        return bboxes, action

    def _crop_and_pad(self, frame, bbox, r):
        h, w, _ = frame.shape
        x_min, y_min, x_max, y_max = bbox

        veh_w, veh_h = (x_max - x_min) * r, (y_max - y_min) * r
        cx, cy = x_min + (x_max - x_min) // 2, y_min + (y_max - y_min) // 2

        square_size = int(max(veh_w, veh_h))

        new_x_min, new_y_min = cx - square_size // 2, cy - square_size // 2
        new_x_max, new_y_max = cx + square_size // 2, cy + square_size // 2

        pad_left, pad_top = max(0, -new_x_min), max(0, -new_y_min)
        pad_right, pad_bottom = max(0, new_x_max - w), max(0, new_y_max - h)

        valid_x_min, valid_y_min = max(0, new_x_min), max(0, new_y_min)
        valid_x_max, valid_y_max = min(w, new_x_max), min(h, new_y_max)

        cropped_frame = frame[valid_y_min:valid_y_max, valid_x_min:valid_x_max]

        if pad_left > 0 or pad_top > 0 or pad_right > 0 or pad_bottom > 0:
            cropped_frame = np.pad(cropped_frame,
                                   ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)),
                                   mode='constant', constant_values=0)

        return cv2.resize(cropped_frame, self.resize)

    def __getitem__(self, idx):
        file_name = self.file_names[idx]
        mp4_path = os.path.join(self.data_dir, f"{file_name}.mp4")
        txt_path = os.path.join(self.data_dir, f"{file_name}.txt")

        bboxes, action = self._parse_annotation(txt_path)

        if action is not None:
            class_str, target_id, start_f = action[0], int(float(action[1])), int(float(action[2]))
        else:
            class_str = 'S'
            target_id = 0
            start_f = 0

        label = 1 if class_str == 'A' else 0

        if target_id not in bboxes:
            target_id = list(bboxes.keys())[0] if bboxes else 0
            target_bbox = bboxes.get(target_id, [0, 0, 224, 224])
        else:
            target_bbox = bboxes[target_id]

        cap = cv2.VideoCapture(mp4_path)
        frames = []
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_f)

        for _ in range(self.clip_length):
            ret, frame = cap.read()
            if not ret: break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(self._crop_and_pad(frame, target_bbox, self.r_value))
        cap.release()

        while len(frames) < self.clip_length:
            frames.append(frames[-1] if frames else np.zeros((self.resize[1], self.resize[0], 3), dtype=np.uint8))

        tensor_frames = [self.transform(f) for f in frames]
        video_tensor = torch.stack(tensor_frames).permute(1, 0, 2, 3)

        return video_tensor, torch.tensor(label, dtype=torch.long)

In [ ]:
# ==========================================
# [2단계] I3D 기반 3D-CNN 모델 설계
# ==========================================
class BasicConv3d(nn.Module):
    def __init__(self, in_channels, out_channels, **kwargs):
        super(BasicConv3d, self).__init__()
        self.conv = nn.Conv3d(in_channels, out_channels, bias=False, **kwargs)
        self.bn = nn.BatchNorm3d(out_channels)

    def forward(self, x):
        return F.relu(self.bn(self.conv(x)), inplace=True)

class InceptionModule3D(nn.Module):
    def __init__(self, in_channels, out_1x1, red_3x3, out_3x3, red_3x3_2, out_3x3_2, out_pool):
        super(InceptionModule3D, self).__init__()
        self.branch1 = BasicConv3d(in_channels, out_1x1, kernel_size=1)
        self.branch2 = nn.Sequential(BasicConv3d(in_channels, red_3x3, kernel_size=1), BasicConv3d(red_3x3, out_3x3, kernel_size=3, padding=1))
        self.branch3 = nn.Sequential(BasicConv3d(in_channels, red_3x3_2, kernel_size=1), BasicConv3d(red_3x3_2, out_3x3_2, kernel_size=3, padding=1))
        self.branch4 = nn.Sequential(nn.MaxPool3d(kernel_size=3, stride=1, padding=1), BasicConv3d(in_channels, out_pool, kernel_size=1))

    def forward(self, x):
        return torch.cat([self.branch1(x), self.branch2(x), self.branch3(x), self.branch4(x)], dim=1)

class HitAndRun3DCNN(nn.Module):
    def __init__(self, num_classes=2):
        super(HitAndRun3DCNN, self).__init__()
        self.conv1 = BasicConv3d(3, 64, kernel_size=7, stride=2, padding=3)
        self.maxpool1 = nn.MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1))
        self.conv2 = BasicConv3d(64, 64, kernel_size=1)
        self.conv3 = BasicConv3d(64, 192, kernel_size=3, padding=1)
        self.maxpool2 = nn.MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1))
        self.inception1 = InceptionModule3D(192, 64, 96, 128, 16, 32, 32)
        self.inception2 = InceptionModule3D(256, 128, 128, 192, 32, 96, 64)
        self.maxpool3 = nn.MaxPool3d(kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1))
        self.inception3 = InceptionModule3D(480, 192, 96, 208, 16, 48, 64)

        self.avg_pool = nn.AdaptiveAvgPool3d((1, 1, 1))
        self.head_conv = nn.Conv3d(512, num_classes, kernel_size=(1, 1, 1))
        self._initialize_weights()

    def forward(self, x):
        x = self.maxpool1(self.conv1(x))
        x = self.maxpool2(self.conv3(self.conv2(x)))
        x = self.maxpool3(self.inception2(self.inception1(x)))
        x = self.avg_pool(self.inception3(x))
        logits = self.head_conv(x)
        return logits.view(logits.size(0), -1)

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm3d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

In [ ]:
# ==========================================
# [3단계] 조기 종료 및 고속 학습 루프 실행부 (AMP 적용)
# ==========================================
# ⚡ [최적화 반영] FP16 혼합 정밀도 학습(Mixed Precision)을 위한 라이브러리 임포트
# Deprecated된 torch.cuda.amp 대신 torch.amp 사용
from torch.amp import autocast, GradScaler

class EarlyStopping:
    def __init__(self, patience=15, delta=0, path='best_model.pth'): # patience를 15로 변경
        self.patience = patience
        self.delta = delta
        self.path = path
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = float('inf')

    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            print(f'조기 종료 카운트: {self.counter} / {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        '''검증 손실이 감소하면 모델을 저장합니다.'''
        if val_loss < self.val_loss_min:
            print(f'검증 손실 감소 ({self.val_loss_min:.6f} --> {val_loss:.6f}). 모델 저장 중...')
            torch.save(model.state_dict(), self.path)
            self.val_loss_min = val_loss

def train_model_optimized(data_dir):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"✅ 사용 중인 디바이스: {device}")

    # 고속 로컬 캐싱 경로 적용 권장 (/content/dataset)
    full_dataset = HitAndRunDataset(data_dir=data_dir, clip_length=30, r_value=1.0)
    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_subset, val_subset = random_split(full_dataset, [train_size, val_size])

    # ⚡ [최적화 반영] persistent_workers=True 및 pin_memory=True 설정을 적용하여 CPU와 GPU 사이의 데이터 전송 지연 극소화
    train_loader = DataLoader(train_subset, batch_size=16, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
    val_loader = DataLoader(val_subset, batch_size=16, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)

    model = HitAndRun3DCNN(num_classes=2).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.00001)

    # ⚡ [최적화 반영] Mixed Precision 손실 스케일러(GradScaler) 객체 생성
    scaler = GradScaler(device="cuda") # 'cuda' 디바이스 명시

    save_path = '/content/drive/MyDrive/Dataset/best_hitandrun_model.pth' # <-- 이 부분을 변경하여 모델 저장 경로 지정
    early_stopping = EarlyStopping(patience=15, path=save_path)
    num_epochs = 50 # 에폭(epochs) 수를 50으로 변경

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        for inputs, labels in train_loader:
            # ⚡ [최적화 반영] non_blocking=True를 활용한 GPU 고속 전송
            inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)

            # ⚡ [최적화 반영] set_to_none=True로 기울기 초기화 시 메모리 가비지 컬렉터 부담 경감
            optimizer.zero_grad(set_to_none=True)

            # ⚡ [최적화 반영] FP16 Mixed Precision 연산 수행 (메모리 사용 절반, 연산 가속)
            with autocast(device_type="cuda"):
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            # ⚡ [최적화 반영] 손실 값 스케일링 후 역전파
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item() * inputs.size(0)

        avg_train_loss = train_loss / len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        correct = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
                # ⚡ [최적화 반영] 평가 검증 시에도 autocast() 연산 적용하여 속도 극대화
                with autocast(device_type="cuda"):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)

                val_loss += loss.item() * inputs.size(0)
                _, preds = torch.max(outputs, 1)
                correct += torch.sum(preds == labels.data)

        avg_val_loss = val_loss / len(val_loader.dataset)
        val_acc = correct.double() / len(val_loader.dataset)

        print(f'Epoch [{epoch+1}/{num_epochs}] Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}')

        early_stopping(avg_val_loss, model)
        if early_stopping.early_stop:
            print("🛑 조기 종료 조건 충족. 학습을 중단합니다.")
            break

    model.load_state_dict(torch.load(save_path))
    return model

# --- 실제 학습을 시작하는 부분 ---
# ⚡ [최적화 반영] Google Colab 로컬 VM의 초고속 NVMe 영역 경로 우선 타겟팅
DATA_DIR = '/content/dataset' # Colab 로컬 디스크 (데이터셋 경로 지정)
GDRIVE_DATA_DIR = '/content/drive/MyDrive/Dataset/Dataset' # 구글 드라이브 기본 경로 (데이터셋 경로 지정)

# ⚡ [최적화 반영] 구글 드라이브(극심한 I/O 병목)에서 로컬 NVMe로 데이터셋 zip 파일을 고속 전송 및 자동 압축 해제하는 파이프라인
if not os.path.exists(DATA_DIR):
    import zipfile
    DRIVE_ZIP = '/content/drive/MyDrive/Dataset/parking-vehicle-hit-an-run-dataset.zip' # <-- 이 부분을 변경하여 데이터셋 ZIP 파일 경로 지정
    if os.path.exists(DRIVE_ZIP):
        print("⚡ 로컬 NVMe 스토리지로 데이터셋 고속 복사 및 압축 해제 중...")
        os.makedirs(DATA_DIR, exist_ok=True)
        with zipfile.ZipFile(DRIVE_ZIP, 'r') as zip_ref:
            zip_ref.extractall(DATA_DIR)
        print("✅ 압축 해제 완료!")
    else:
        # zip 파일이 없을 때만 차선책으로 구글 드라이브 직접 마운트 경로 사용
        DATA_DIR = GDRIVE_DATA_DIR

if os.path.exists(DATA_DIR) and len(os.listdir(DATA_DIR)) > 0:
    print(f"데이터 디렉토리 확인 완료: {DATA_DIR}. 학습을 시작합니다!")
    trained_model = train_model_optimized(data_dir=DATA_DIR)
else:
    print(f"⚠️ 데이터 폴더를 찾을 수 없습니다: {DATA_DIR}")

In [ ]:
import cv2
import torch
import numpy as np
import torchvision.transforms as transforms
import torch.nn.functional as F
import os

# ==========================================
# 1. CAM 출력을 위한 특징 맵 추출기 (Hook)
# ==========================================
activation = {}
def get_activation(name):
    def hook(model, input, output):
        activation[name] = output.detach()
    return hook

# ========================================== # 사용자 입력 파일 디렉토리 관련 부분
# 2. 정사각형 패딩 + 라벨 프리 + 전체 화면 CAM 예측 함수
# ==========================================
def predict_hit_and_run_final(model, video_path, txt_path, target_id=0, r_value=1.0, resize=(224, 224)):
    # `video_path`: 예측에 사용할 입력 비디오 파일의 경로입니다.
    # `txt_path`: 비디오에 해당하는 어노테이션 텍스트 파일의 경로입니다.
    print(f"[{os.path.basename(video_path)}] 전체 화면 분석 시작...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()

    # 모델의 마지막 특징 맵(inception3) 추출을 위한 Hook 등록
    handle = model.inception3.register_forward_hook(get_activation('inception3'))

    # 1. 텍스트 파일 파싱 (A/S 라벨 없이 car 좌표만 수집)
    bboxes = {}
    if not os.path.exists(txt_path):
        print(f"⚠️ 텍스트 파일을 찾을 수 없습니다: {txt_path}")
        return

    with open(txt_path, 'r') as f:
        for line in f.readlines():
            parts = line.strip().split(',')
            if len(parts) >= 6 and parts[0] == 'car':
                bboxes[int(parts[1])] = [int(parts[2]), int(parts[3]), int(parts[4]), int(parts[5])]

    if target_id not in bboxes:
        print(f"⚠️ ID {target_id}번 차량의 좌표가 없습니다. 존재하는 ID: {list(bboxes.keys())}")
        return

    target_bbox = bboxes[target_id]

    # 2. 비율 왜곡 방지 정사각형 패딩 함수
    def crop_square_and_pad(frame, bbox, r):
        h, w, _ = frame.shape
        x_min, y_min, x_max, y_max = bbox
        vw, vh = (x_max - x_min) * r, (y_max - y_min) * r
        cx, cy = x_min + (x_max - x_min) // 2, y_min + (y_max - y_min) // 2

        # 가로/세로 중 긴 변을 기준으로 정사각형(Square) 크기 결정
        side = int(max(vw, vh))
        nx1, ny1 = cx - side // 2, cy - side // 2
        nx2, ny2 = cx + side // 2, cy + side // 2

        v_x1, v_y1 = max(0, nx1), max(0, ny1)
        v_x2, v_y2 = min(w, nx2), min(h, ny2)
        p_l, p_t = max(0, -nx1), max(0, -ny1)
        p_r, p_b = max(0, nx2 - w), max(0, ny2 - h)

        cropped = frame[v_y1:v_y2, v_x1:v_x2]
        if p_l > 0 or p_t > 0 or p_r > 0 or p_b > 0:
            cropped = np.pad(cropped, ((p_t, p_b), (p_l, p_r), (0, 0)), mode='constant')
        return cv2.resize(cropped, resize), (nx1, ny1, nx2, ny2)

    # 3. 비디오 로드 및 전처리
    cap = cv2.VideoCapture(video_path)
    original_full_frames = []
    tensor_frames = []
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # 역매핑을 위한 첫 프레임 좌표계 계산
    ret, first_frame = cap.read()
    if not ret: return
    _, (rx1, ry1, rx2, ry2) = crop_square_and_pad(first_frame, target_bbox, r_value)
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

    while True:
        ret, frame = cap.read()
        if not ret: break
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        original_full_frames.append(frame_rgb)
        processed, _ = crop_square_and_pad(frame_rgb, target_bbox, r_value)
        tensor_frames.append(transform(processed))
    cap.release()

    full_video_tensor = torch.stack(tensor_frames).permute(1, 0, 2, 3)
    orig_h, orig_w = original_full_frames[0].shape[:2]

    # 4. 출력 비디오 저장 설정
    out_path = f'/content/drive/MyDrive/Dataset/Predictions/Result/final_{os.path.basename(video_path)}' # 예측 결과 비디오 저장 경로
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    out_video = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), 30.0, (orig_w, orig_h))

    # 실제 화면에 표시될 유효 좌표
    v_nx1, v_ny1 = max(0, rx1), max(0, ry1)
    v_nx2, v_ny2 = min(orig_w, rx2), min(orig_h, ry2)

    print("🔍 슬라이딩 윈도우 추론 및 히트맵 생성 중...")
    with torch.no_grad():
        for i in range(len(tensor_frames)):
            if i < 29: # 첫 29프레임은 이전 기록이 부족하여 원본 뼈대만 출력
                final_frame = cv2.cvtColor(original_full_frames[i], cv2.COLOR_RGB2BGR)
                cv2.rectangle(final_frame, (v_nx1, v_ny1), (v_nx2, v_ny2), (0, 255, 0), 2)
            else:
                # 30프레임 슬라이딩 윈도우 적용
                clip = full_video_tensor[:, i-29:i+1, :, :].unsqueeze(0).to(device)
                outputs = model(clip)
                probs = F.softmax(outputs, dim=1)
                pred_class = torch.argmax(probs, dim=1).item()
                conf = probs[0][pred_class].item() * 100

                # CAM 계산 (가중치 * 특징 맵)
                feat_map = activation['inception3'].squeeze(0)
                weight = model.head_conv.weight[pred_class]
                cam = torch.sum(weight * feat_map, dim=0)
                cam = F.relu(cam)
                cam_2d = torch.mean(cam, dim=0).cpu().numpy()
                cam_2d = (cam_2d - cam_2d.min()) / (cam_2d.max() - cam_2d.min() + 1e-8)

                # 히트맵 원본 해상도로 복원 및 자르기
                heatmap = cv2.applyColorMap(np.uint8(255 * cam_2d), cv2.COLORMAP_JET)
                heatmap = cv2.resize(heatmap, (rx2-rx1, ry2-ry1))
                heatmap_valid = heatmap[v_ny1-ry1 : (v_ny1-ry1)+(v_ny2-v_ny1), v_nx1-rx1 : (v_nx1-rx1)+(v_nx2-v_nx1)]

                final_frame = cv2.cvtColor(original_full_frames[i], cv2.COLOR_RGB2BGR)
                roi = final_frame[v_ny1:v_ny2, v_nx1:v_nx2]

                # 충돌 감지 여부에 따른 시각화 처리
                if pred_class == 1: # 충돌(A)
                    final_frame[v_ny1:v_ny2, v_nx1:v_nx2] = cv2.addWeighted(roi, 0.6, heatmap_valid, 0.4, 0)
                    color, label_text = (0, 0, 255), f"Accident ({conf:.1f}%)"
                else: # 정상(S)
                    color, label_text = (0, 255, 0), f"Normal ({conf:.1f}%)"

                cv2.rectangle(final_frame, (v_nx1, v_ny1), (v_nx2, v_ny2), color, 3)
                cv2.putText(final_frame, label_text, (50, 70), cv2.FONT_HERSHEY_SIMPLEX, 1.5, color, 3)

            out_video.write(final_frame)

    out_video.release()
    handle.remove()
    print(f"✨ 분석 완료! 결과 파일: {out_path}")

# ==========================================
# 3. [핵심] 가중치 불러오기 및 실행부
# ==========================================
# ※ 주의: 이전에 HitAndRun3DCNN 클래스 코드가 실행되어 있어야 합니다.

# 저장된 가중치 파일 경로 지정
WEIGHTS_PATH = '/content/drive/MyDrive/Dataset/best_hitandrun_model.pth' # 모델 가중치 파일 경로

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. 빈 모델 객체(껍데기) 생성
loaded_model = HitAndRun3DCNN(num_classes=2).to(device)

try:
    # 2. 구글 드라이브에 저장된 가중치(.pth) 파일을 빈 모델에 덮어쓰기
    loaded_model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
    print("✅ 성공적으로 학습된 가중치(best_hitandrun_model.pth)를 불러왔습니다!")

    # 3. 테스트할 타겟 영상과 텍스트 경로 지정
    TARGET_VIDEO_PATH = '/content/drive/MyDrive/Dataset/Predictions/Original/220510_LA_0001.mp4' # 타겟 비디오 파일 경로
    TARGET_TXT_PATH = '/content/drive/MyDrive/Dataset/Predictions/txt/220510_LA_0001.txt' # 타겟 텍스트 파일 경로

    # 4. 가중치가 장착된 모델을 활용하여 예측 실행
    # target_id=0 은 텍스트 파일의 0번 차량을 추적하겠다는 의미입니다.
    predict_hit_and_run_final(loaded_model, TARGET_VIDEO_PATH, TARGET_TXT_PATH, target_id=0)

except FileNotFoundError:
    print(f"⚠️ 에러: {WEIGHTS_PATH} 경로에 가중치 파일이 없습니다.")
    print("경로가 정확한지 확인하시거나, 먼저 학습 코드를 실행하여 가중치 파일을 생성해 주세요.")

✅ 성공적으로 학습된 가중치(best_hitandrun_model.pth)를 불러왔습니다!
[220510_LA_0001.mp4] 전체 화면 분석 시작...
⚠️ 텍스트 파일을 찾을 수 없습니다: /content/drive/MyDrive/Dataset/Predictions/txt/220510_LA_0001.txt


In [ ]:
# ==========================================
# [최적화] 배치 추론 및 Stride=2(2프레임 건너뛰기) 적용 고속 평가
# ==========================================
import os
import cv2
import torch
import numpy as np
import torchvision.transforms as transforms
import torch.nn.functional as F
import random
from tqdm import tqdm
from torch.cuda.amp import autocast

# ⚡ [최적화 반영] evaluate_folder_accuracy_optimized 함수 정의 (stride 및 eval_batch_size 매개변수 도입)
def evaluate_folder_accuracy_optimized(model, folder_path, target_id=0, r_value=1.0, resize=(224, 224), num_samples=None, stride=2, eval_batch_size=32):
    print(f"📂 [{folder_path}] 폴더의 영상 최적화 평가를 준비합니다... (Stride: {stride}, Batch Size: {eval_batch_size})")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    def crop_square_and_pad(frame, bbox, r):
        h, w, _ = frame.shape
        x_min, y_min, x_max, y_max = bbox
        vw, vh = (x_max - x_min) * r, (y_max - y_min) * r
        cx, cy = x_min + (x_max - x_min) // 2, y_min + (y_max - y_min) // 2
        side = int(max(vw, vh))
        nx1, ny1 = cx - side // 2, cy - side // 2
        nx2, ny2 = cx + side // 2, cy + side // 2

        v_x1, v_y1 = max(0, nx1), max(0, ny1)
        v_x2, v_y2 = min(w, nx2), min(h, ny2)
        p_l, p_t = max(0, -nx1), max(0, -ny1)
        p_r, p_b = max(0, nx2 - w), max(0, ny2 - h)

        cropped = frame[v_y1:v_y2, v_x1:v_x2]
        if p_l > 0 or p_t > 0 or p_r > 0 or p_b > 0:
            cropped = np.pad(cropped, ((p_t, p_b), (p_l, p_r), (0, 0)), mode='constant')
        return cv2.resize(cropped, resize)

    all_files = os.listdir(folder_path)
    mp4_files = [f for f in all_files if f.endswith('.mp4')]
    valid_pairs = [mp4 for mp4 in mp4_files if mp4.replace('.mp4', '.txt') in all_files]

    if not valid_pairs:
        print("⚠️ 평가할 수 있는 영상-텍스트 짝이 없습니다.")
        return

    if num_samples is not None:
        actual_samples = min(num_samples, len(valid_pairs))
        selected_files = random.sample(valid_pairs, actual_samples)
        print(f"🎲 총 {len(valid_pairs)}개의 영상 쌍 중에서 {actual_samples}개를 추출하여 최적화 추론을 시작합니다.")
    else:
        selected_files = valid_pairs
        print(f"전체 {len(valid_pairs)}개의 영상 쌍을 모두 최적화 평가합니다.")

    total_videos = 0
    correct_preds = 0
    wrong_list = []

    with torch.no_grad():
        for mp4_file in tqdm(selected_files, desc="최적화 평가 진행률"):
            base_name = mp4_file.replace('.mp4', '')
            video_path = os.path.join(folder_path, mp4_file)
            txt_path = os.path.join(folder_path, f"{base_name}.txt")

            # 실제 정답 추출
            parts = base_name.split('_')
            is_accident_gt = (parts[1][1] == 'A') if (len(parts) >= 2 and len(parts[1]) == 2) else False
            gt_label = 1 if is_accident_gt else 0

            # 0번 차량 좌표 파싱
            bboxes = {}
            with open(txt_path, 'r') as f:
                for line in f.readlines():
                    l_parts = line.strip().split(',')
                    if len(l_parts) >= 6 and l_parts[0] == 'car':
                        bboxes[int(l_parts[1])] = [int(l_parts[2]), int(l_parts[3]), int(l_parts[4]), int(l_parts[5])]

            if target_id not in bboxes:
                if not bboxes: continue
                target_bbox = list(bboxes.values())[0]
            else:
                target_bbox = bboxes[target_id]

            # 비디오 로드 및 고속 프레임 크롭
            cap = cv2.VideoCapture(video_path)
            tensor_frames = []
            while True:
                ret, frame = cap.read()
                if not ret: break
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                processed = crop_square_and_pad(frame_rgb, target_bbox, r_value)
                tensor_frames.append(transform(processed))
            cap.release()

            while len(tensor_frames) < 30:
                tensor_frames.append(tensor_frames[-1] if tensor_frames else torch.zeros((3, 224, 224)))

            full_video_tensor = torch.stack(tensor_frames).permute(1, 0, 2, 3)

            # ⚡ [최적화 반영] Stride=2 설정을 도입하여 슬라이딩 윈도우 3D 클립 조각들을 2프레임 간격으로 구성 (계산량 50% 단축)
            clips = []
            for i in range(0, len(tensor_frames) - 29, stride):
                clip = full_video_tensor[:, i:i+30, :, :]
                clips.append(clip)

            if not clips:
                clips.append(full_video_tensor[:, :30, :, :])

            # 배치 구성 및 GPU 전송
            clips_tensor = torch.stack(clips).to(device)

            # ⚡ [최적화 반영] 배치 추론 (Batch Inference) 기법으로 GPU 병렬 처리 극대화 (B, C, T, H, W)
            predicted_label = 0
            num_clips = clips_tensor.size(0)
            for b_idx in range(0, num_clips, eval_batch_size):
                batch_clips = clips_tensor[b_idx:b_idx+eval_batch_size]

                # ⚡ [최적화 반영] 추론 가속을 위한 autocast() 영역 설정
                with autocast():
                    outputs = model(batch_clips)
                    probs = F.softmax(outputs, dim=1)
                    pred_classes = torch.argmax(probs, dim=1)

                if (pred_classes == 1).any():
                    predicted_label = 1
                    break

            total_videos += 1
            if predicted_label == gt_label:
                correct_preds += 1
            else:
                wrong_list.append({
                    "file": mp4_file,
                    "gt": "Accident(충돌)" if gt_label == 1 else "Normal(정상)",
                    "pred": "Accident(충돌)" if predicted_label == 1 else "Normal(정상)"
                })

    accuracy = (correct_preds / total_videos) * 100 if total_videos > 0 else 0
    print("\n" + "="*50)
    print("📊 [모델 최적화 성능 평가 결과 (Stride=2)]")
    print("="*50)
    print(f"총 평가 영상 수 : {total_videos} 개")
    print(f"정답 맞춘 수    : {correct_preds} 개")
    print(f"최종 Accuracy   : {accuracy:.2f}%")
    print("="*50)

    if wrong_list:
        print("\n❌ [오답 노트 (틀린 영상 리스트)]")
        for w in wrong_list:
            print(f" - {w['file']} (실제: {w['gt']}  |  모델예측: {w['pred']})")
    else:
        print("\n🎉 모든 영상을 완벽하게 맞췄습니다!")

# --- 평가 실행부 ---
WEIGHTS_PATH = '/content/drive/MyDrive/capstone-26/hitandrun_model_best.pth'
TEST_FOLDER_PATH = '/content/dataset'
NUMBER_OF_SAMPLES = 50

loaded_model = HitAndRun3DCNN(num_classes=2).to(device)

try:
    loaded_model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
    print("✅ 성공적으로 모델 가중치를 불러왔습니다!")
    # ⚡ [최적화 반영] Stride=2 를 적용한 초고속 폴더 정확도 평가 실행
    evaluate_folder_accuracy_optimized(loaded_model, TEST_FOLDER_PATH, target_id=0, num_samples=NUMBER_OF_SAMPLES, stride=2)
except FileNotFoundError:
    print(f"⚠️ 에러: {WEIGHTS_PATH} 경로에 가중치 파일이 없습니다. 먼저 모델을 학습시켜주세요.")